In [ ]:
"""
<RAG(Retrieval Augmented Generation)>
 - "검색 증강 생성"이라고 칭함
 - RAG = 대규모 언어 모델과
          + 외부 데이터(문서파일, 웹데이터, 벡터 데이터베이스)의 정보에서
            검색한 유사 정보를 결합하여
       = 정확하고 신뢰성 높은 결과를 생성하는 기술


 - 외부 데이터베이스 생성 방법
   1. 요청(질문) 주제에 맞는 -> 문서 로더
   2. 로드된 문서를 벡터 데이터베이스에 저장
      - 벡터 데이터베이스(행과 열의 묶음이 여러개 있는 데이터를 벡터 데이터 베이스 라고 함)
        : 크로마(Chroma) 또는 파이스(FAISS) 외에 다양한 벡터 DB가 있음


<문서 로더(Document Loader)>
 * 랭체인의 문서 로더는 다양한 데이터 소스와 형식을
   - 랭체인 프레임워크에서 사용할 수 있도록
   - 표준화된 문서 형식으로 변환하는 도구임


 * 80개 이상의 다양한 유형을 지원함
   - 텍스트, PDF, 웹페이지, CSV 등 다양한 형식의 데이터를 처리할 수 있음


 * 모든 로더는 데이터를 content와 metadata를 포함하는
   - 표준 문서 객체로 변환하여 랭체인의 다른 컴포넌트와 쉽게 통합할 수 있음


<문서 로더의 클래스(class) 종류>
 * WebBaseLoader
   - 웹 페이지(HTML 데이터)의 내용을 로드하는 기능을 제공함
 * PyPDFLoader
   - PDF 파일을 로드하고 필요에 따라 페이지 단위로 분할하는 기능을 제공함
 * CSVLoader
   - CSV 파일을 로드하는 기능을 제공함
 * DirectoryLoader
   - 지정된 디렉토리 내의 모든 파일을 로드하는 기능을 제공함
 * UnstructuredLoader
   - 다양한 형식의 파일을 로드하는 기능을 제공함


<문서 로더 라이브러리 설치>
  * Web, PDF, CSV 등등 파일 및 폴더 로드 처리 (이미 설치되어 있음)
    >uv pip install langchain_community


  * 로드된 웹 페이지 데이터 처리 라이브러리(이미 설치된 경우 안해도 됨)
    >uv pip install beautifulsoup4
"""


### 웹 페이지 읽어들이기(WebBaseLoader)

#### 라이브러리 정의

In [ ]:
"""
* 웹페이지 로더 라이브러리를 설치후
 - 단독으로, from langchain_community.document_loaders import WebBaseLoader 불러들일 경우
 - 아래 <경고> 메시지가 나타남
 
<경고 내용>
 - USER_AGENT environment variable not set, consider setting it to identify your requests.
 - USER_AGENT 환경 변수가 설정되지 않았습니다. 요청을 식별하기 위해 설정하는 것이 좋습니다.


<경고가 발생한 이유>
 - 웹 페이지를 로드하기 위해서는 현재 프로그램 영역에서
   외부 웹페이지(웹서버) 영역으로 요청(URL)을 보내 후
   응답(페이지 정보)을 받아오게 됨
 - 외부로 요청을 보내는 일련의 과정을 OS에 식별자로 정의해야
   요청에 대한 추적이 용이하다는 의견임


<해결 방법>
 - 경고 메시지의 "USER_AGENT"라는 이름으로 환경변수에 설정하라는 의미임
 - 즉, 사용자 에이전트 설정을 하라는 의미임  
"""


In [2]:
### 사용 운영체제 라이브러리
import os

### 웹페이지 정보수집을 위한 에이전트 설정
# - 웹페이지 로더 라이브러리를 사용하기 위해서는 사전에 에이전트를 설정할 필요 있음
os.environ["USER_AGENT"] = "MyApp/1.0 (Custom LangChain Application)"

### 웹페이지 로더 라이브러리
from langchain_community.document_loaders import WebBaseLoader

### 단일 URL 요청 처리하기

In [4]:
### 요청 URL 변수로 정의
url = "https://docs.smith.langchain.com/"

### 웹페이지 로더 클래스 생성하기 : 요청 url을 넣어 주셔야 합니다.
loader_single_page = WebBaseLoader(url)
loader_single_page

### 단일 웹페이지 URL 정보에 대한 페이지 정보 로드(읽어들이기)
# - 반환 타입은 리스트(list) 타입
single_doc = loader_single_page.load()
single_doc

[Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failure

In [ ]:
### 읽어들인 웹페이지 정보는 리스트 타입으로 되어있음
# - 따라서, 리스트 안으로 들어가서 데이터 추출하기
single_doc[0]

### 읽어들인 정보에는 metadata와 page_content 속성 2개가 있음
# - metadata : 웹페이지 기본 정보가 들어있음(딕셔너리 타입으로 정보가 있음)
# - page_content : 웹페이지의 본문 내용이 들어있음

Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures

In [29]:
### metadata(웹페이지 기본 정보) 확인하기
meta_dict = single_doc[0].metadata

### metadata 내에 있는 모든 정보를 출력(print) 하기
for k in meta_dict.items():
    print(f" [{k}] : {v}")


# 딕셔너리에서 키값만 출력하기
print(f"딕셔너리의 key값만 추출하기 : {meta_dict.keys()}")
# 딕셔너리에서 밸류값만 출력하기
print(f"딕셔너리의 value값만 추출하기 : {meta_dict.values()}")
# 딕셔너리에서 키값 밸류값 모두 출력하기
print(f"딕셔너리의 키값 밸류값 추출하기 : {meta_dict.items()}")

 [('source', 'https://docs.smith.langchain.com/')] : en
 [('title', 'LangSmith Observability - Docs by LangChain')] : en
 [('description', 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.')] : en
 [('language', 'en')] : en
딕셔너리의 key값만 추출하기 : dict_keys(['source', 'title', 'description', 'language'])
딕셔너리의 value값만 추출하기 : dict_values(['https://docs.smith.langchain.com/', 'LangSmith Observability - Docs by LangChain', 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'en'])
딕셔너리의 키값 밸류값 추출하기 : dict_items([('source', 'https://docs.smith.langchain.com/'), ('title', 'LangSmith Observability - Docs by LangChain'), ('description', 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.'), ('language', 'en')])


In [18]:
### 수집한 페이지 본문 내용 확인하기
# - 본문 내용은 page_content 속성에 문자열로 저장되어 있음
# - page_content 속성(변수)에 저장된 값 출력해주세요...
single_doc

[Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failure

In [34]:
text = single_doc[0].page_content
print(text)






LangSmith Observability - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain

### 새로운 URL 웹페이지 데이터 수집하기

In [ ]:
### 새로운 URL : https://www.ibm.com/kr-ko/think/topics/machine-learning
# - 웹페이지 본문 내용 추출하여 "text 변수에 저장"하여 출력해 주세요..

In [78]:
url = "https://www.ibm.com/kr-ko/think/topics/machine-learning"
url

load_page = WebBaseLoader(url)
load_page

load_doc = load_page.load()
load_doc

text = load_doc[0].page_content
print(text)
















머신 러닝이란 무엇인가요? | IBM 
























































































































                    


  
  
    
    머신 러닝이란 무엇인가요?

  





    


                



                    게시일 2025년 08월 18일
                

                    수정일 2026년 10월 2일
                

















                By 
                Dave Bergmann










        머신 러닝이란 무엇인가요?
    


머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.


머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결과로 변환하는 일반화는 머신 러닝의 기본 목표입니다. 본질적으로 학습된 모델은 학습 데이터에서 학습한 패턴을 적용하여 실제 작업에 대한 올바른 아웃풋을 

In [79]:
### 임의 URL을 전달 받아서, 해당 URL 웹페이지의 본문 내용을 추출하여 반환하는 
# - 함수 정의하기 : 함수 이름은 getPageContent
# - 함수로부터 반환받은 본문 내용 잘 출력되는지 확인까지 해주세요..
def getPageContent(get_url):
    url = get_url

    loader_single_page = WebBaseLoader(url)

    single_doc = loader_single_page.load()

    text = single_doc[0].page_content

    return text

In [80]:
gPC = getPageContent("https://www.ibm.com/kr-ko/think/topics/machine-learning")

print(gPC)
















머신 러닝이란 무엇인가요? | IBM 
























































































































                    


  
  
    
    머신 러닝이란 무엇인가요?

  





    


                



                    게시일 2025년 08월 18일
                

                    수정일 2026년 10월 2일
                

















                By 
                Dave Bergmann










        머신 러닝이란 무엇인가요?
    


머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.


머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결과로 변환하는 일반화는 머신 러닝의 기본 목표입니다. 본질적으로 학습된 모델은 학습 데이터에서 학습한 패턴을 적용하여 실제 작업에 대한 올바른 아웃풋을 

In [109]:
### text 데이터에서 \n, 문장의 양끝 공백, 탭 등 제거해 주세요...
text = text.replace("\n","")

text = text.replace("\t","")
print(text)

머신러닝이란무엇인가요?|IBM머신러닝이란무엇인가요?게시일2025년08월18일수정일2026년10월2일ByDaveBergmann머신러닝이란무엇인가요?머신러닝은학습데이터의패턴을"학습"한후새로운데이터에대해정확한추론을할수있는알고리즘에초점을맞춘인공지능(AI)의하위집합입니다.이패턴인식기능을통해머신러닝모델은명시적이고하드코딩된지침없이결정이나예측을내릴수있습니다.머신러닝은예측모델부터자율주행 차량,대규모언어모델(LLM)및기타 생성형AI도구에이르기까지대부분의최신AI시스템의중추를제공하는AI분야를지배하게되었습니다.머신러닝(ML)의핵심적인전제는모델학습이라는프로세스를통해모델이사용될실제문제와충분히유사한작업데이터세트에서모델의성능을최적화하면모델이궁극적인사용사례에서보게되는새데이터를정확하게예측할수있다는것입니다.학습자체는단순히목적을위한수단일뿐이며,학습데이터의성능을실제시나리오에서유용한결과로변환하는일반화는머신러닝의기본목표입니다. 본질적으로학습된모델은학습데이터에서학습한패턴을적용하여실제작업에대한올바른아웃풋을추론하는것입니다.따라서AI모델의배포를 AI추론이라고합니다.대규모또는'심층'인공신경망에의해구동되는머신러닝의하위집합인딥러닝은지난수십년동안AI가활용되는거의모든영역에서최첨단AI모델아키텍처로등장했습니다.기존머신러닝의명시적으로정의된알고리즘과달리,딥러닝은매우복잡한데이터의복잡한뉘앙스를학습할수있는최상의능력을제공하는분산된수학연산"네트워크"에의존합니다.딥러닝에는매우많은양의데이터와계산리소스가필요하기때문에딥러닝의출현은"빅데이터"와그래픽처리장치(GPU)의중요성이높아진것과일치합니다.머신러닝분야는데이터과학분야와밀접하게얽혀있습니다.어떤의미에서머신러닝은데이터분석을자동화하고(더중요하게는)해당분석에서학습한내용을관련작업의자율적실행에적용하기위한알고리즘및기술의모음으로이해할수있습니다.(핵심개념자체는아니지만)이용어의어원은많은경우ArthurL.Samuel이1959년에IBM저널에기고한'체커게임을활용한몇가지머신러닝연구'로여겨집니다.논문의서문에서Samuel은머신러닝의이상적인결과를깔끔하게설명합니다:"컴퓨터는프로그램을작성한사람이할수있는것보다더나은체커게임을하는법